# Stage 4 — cell detection

Detection uses the canonical source entry point and retains visual inspection.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks" or PROJECT_ROOT.parent.name == "notebooks":
    while PROJECT_ROOT.name != "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from src.io import PipelinePaths

SAMPLE_ID = "44b6_0113de3b"
FRAME = 0
paths = PipelinePaths.discover(PROJECT_ROOT)
sample_path = paths.sample_zarr(SAMPLE_ID)


In [ ]:
import matplotlib.pyplot as plt
from src.api import create_binary_mask, detect_cells, preprocess_volume, segment_instances
from src.io import load_timepoint

raw = load_timepoint(sample_path, FRAME)
preprocessed = preprocess_volume(raw)
labels = segment_instances(create_binary_mask(preprocessed))
cells, trace = detect_cells(labels, return_diagnostics=True)
cells.head()


In [ ]:
z = labels.shape[0] // 2
visible = cells[(cells.centroid_z >= z - 0.5) & (cells.centroid_z <= z + 0.5)]
plt.figure(figsize=(8, 8)); plt.imshow(labels[z], cmap="gray")
plt.scatter(visible.centroid_x, visible.centroid_y, s=20, facecolors="none", edgecolors="red")
plt.title(f"{len(cells)} detected cells")
